# Actor-Critic Methods

## Learning Objectives
1. Implement tabular actor-critic using a Q-table critic and softmax actor on GridWorld
2. Build neural A2C with shared network and GAE on CartPole simulation
3. Apply actor-critic to continuous control (pendulum) and multi-step returns
4. Compare REINFORCE vs A2C vs A2C+entropy on sample efficiency

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch: {torch.__version__}')

## Level 1: Tabular Actor-Critic on GridWorld

Critic = Q-table (tabular state-value V(s)).  
Actor = softmax policy parameterized by preference table theta.  
Shows the core actor-critic loop without neural network complexity.

In [ ]:
class GridWorld:
    """4x4 GridWorld: start=(0,0), goal=(3,3), wall=(1,1)."""

    def __init__(self, size: int = 4):
        self.size = size
        self.goal = (size - 1, size - 1)
        self.wall = (1, 1)
        self.n_actions = 4
        self.n_states = size * size
        self.action_deltas = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        self.reset()

    def reset(self) -> int:
        self.pos = (0, 0)
        return self._idx(self.pos)

    def _idx(self, pos) -> int:
        return pos[0] * self.size + pos[1]

    def step(self, action: int):
        dr, dc = self.action_deltas[action]
        nr, nc = self.pos[0] + dr, self.pos[1] + dc
        if not (0 <= nr < self.size and 0 <= nc < self.size):
            return self._idx(self.pos), -1.0, False
        npos = (nr, nc)
        if npos == self.wall:
            return self._idx(self.pos), -5.0, False
        self.pos = npos
        if self.pos == self.goal:
            return self._idx(self.pos), 10.0, True
        return self._idx(self.pos), -1.0, False


def softmax_policy(theta_s: np.ndarray) -> np.ndarray:
    """Compute action probabilities via softmax over preferences."""
    exp_t = np.exp(theta_s - theta_s.max())
    return exp_t / exp_t.sum()


def run_tabular_actor_critic(
    env: GridWorld,
    n_episodes: int = 500,
    alpha_actor: float = 0.05,   # Actor learning rate
    alpha_critic: float = 0.1,   # Critic (V-value) learning rate
    gamma: float = 0.99,
) -> tuple:
    """Tabular actor-critic. Actor: theta table. Critic: V-value table."""
    # Actor: preference table theta[state, action]
    theta = np.zeros((env.n_states, env.n_actions))
    # Critic: state-value function V(s)
    V = np.zeros(env.n_states)

    episode_rewards = []
    td_error_log = []

    for ep in range(n_episodes):
        state = env.reset()
        total_reward = 0.0
        ep_td = []
        done = False
        steps = 0

        while not done and steps < 200:
            # Actor: sample action from softmax policy
            pi = softmax_policy(theta[state])
            action = int(np.random.choice(env.n_actions, p=pi))

            next_state, reward, done = env.step(action)

            # Critic: compute TD error (advantage estimate)
            td_error = reward + gamma * V[next_state] * (1 - done) - V[state]

            # Critic update: move V(s) toward TD target
            V[state] += alpha_critic * td_error

            # Actor update: increase log-prob of action by td_error amount
            # Gradient of log pi(a|s) w.r.t theta[s, a] = 1 - pi[a] (for selected a)
            # For other actions: -pi[a]
            for a in range(env.n_actions):
                grad = (1.0 if a == action else 0.0) - pi[a]
                theta[state, a] += alpha_actor * td_error * grad

            ep_td.append(abs(td_error))
            total_reward += reward
            state = next_state
            steps += 1

        episode_rewards.append(total_reward)
        td_error_log.append(np.mean(ep_td))

    return theta, V, episode_rewards, td_error_log


env = GridWorld()
theta_final, V_final, tab_ac_rewards, tab_td_errors = run_tabular_actor_critic(env, n_episodes=500)

print(f'Tabular Actor-Critic: mean reward (last 50) = {np.mean(tab_ac_rewards[-50:]):.2f}')
print(f'Mean TD error (last 50 eps): {np.mean(tab_td_errors[-50:]):.4f}')
print(f'\nLearned V(s) for 4x4 grid:')
print(V_final.reshape(4, 4).round(2))

## Level 2: Neural A2C with GAE on CartPole Simulation

Shared base network with separate policy and value heads.  
GAE: A_t = sum_{l>=0} (gamma*lambda)^l * delta_{t+l} for reduced variance.

In [ ]:
def cartpole_step(state: np.ndarray, action: int, dt: float = 0.02):
    """CartPole physics. State: [x, x_dot, theta, theta_dot]."""
    x, x_dot, theta, theta_dot = state
    force = 10.0 if action == 1 else -10.0
    cos_t, sin_t = np.cos(theta), np.sin(theta)
    temp = (force + 0.05 * theta_dot**2 * sin_t) / 1.1
    theta_acc = (9.8 * sin_t - cos_t * temp) / (0.5 * (4/3 - 0.1 * cos_t**2 / 1.1))
    x_acc = temp - 0.05 * theta_acc * cos_t / 1.1
    x += dt * x_dot; x_dot += dt * x_acc
    theta += dt * theta_dot; theta_dot += dt * theta_acc
    done = abs(x) > 2.4 or abs(theta) > 0.2
    return np.array([x, x_dot, theta, theta_dot]), 1.0 if not done else 0.0, done


def cartpole_reset():
    return np.random.uniform(-0.05, 0.05, 4)


class ActorCriticNetwork(nn.Module):
    """Shared-base actor-critic: one base, separate policy and value heads."""

    def __init__(self, state_dim: int, n_actions: int, hidden: int = 128):
        super().__init__()
        # Shared feature extraction layers
        self.base = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
        )
        # Policy head: outputs action log-probabilities
        self.policy_head = nn.Linear(hidden, n_actions)
        # Value head: outputs scalar V(s)
        self.value_head = nn.Linear(hidden, 1)

    def forward(self, x: torch.Tensor) -> tuple:
        h = self.base(x)
        log_probs = F.log_softmax(self.policy_head(h), dim=-1)
        value = self.value_head(h).squeeze(-1)
        return log_probs, value


def compute_gae(
    rewards: list,
    values: list,
    next_value: float,
    dones: list,
    gamma: float = 0.99,
    lam: float = 0.95,
) -> np.ndarray:
    """Compute Generalized Advantage Estimation (GAE). Backward pass over rollout."""
    T = len(rewards)
    advantages = np.zeros(T)
    gae = 0.0

    # Bootstrap from the last state if not terminal
    values_extended = list(values) + [next_value]

    for t in reversed(range(T)):
        # TD error for step t
        delta = rewards[t] + gamma * values_extended[t + 1] * (1 - dones[t]) - values_extended[t]
        # GAE: exponentially weighted sum of future TD errors
        gae = delta + gamma * lam * (1 - dones[t]) * gae
        advantages[t] = gae

    return advantages


def run_a2c(
    n_episodes: int = 400,
    n_steps: int = 20,          # Collect n_steps before each update
    gamma: float = 0.99,
    lam: float = 0.95,          # GAE lambda
    lr: float = 3e-4,
    entropy_coeff: float = 0.01,
    value_coeff: float = 0.5,
) -> tuple:
    """A2C on CartPole simulation. Collects n_steps rollouts then updates."""
    ac = ActorCriticNetwork(4, 2).to(device)
    optimizer = optim.Adam(ac.parameters(), lr=lr)

    episode_rewards = []
    entropy_log = []
    current_episode_reward = 0.0
    state = cartpole_reset()
    ep_count = 0

    # Run for n_episodes worth of data
    total_steps_target = n_episodes * 150  # ~150 steps per episode
    total_steps = 0

    while ep_count < n_episodes:
        # Collect n_steps of experience
        states_buf, actions_buf = [], []
        rewards_buf, values_buf, dones_buf = [], [], []
        log_probs_buf = []

        for _ in range(n_steps):
            s_t = torch.FloatTensor(state).unsqueeze(0).to(device)
            with torch.no_grad():
                log_probs, value = ac(s_t)

            probs = log_probs.exp().squeeze().cpu().numpy()
            action = int(np.random.choice(2, p=probs))
            next_state, reward, done = cartpole_step(state, action)

            states_buf.append(state.copy())
            actions_buf.append(action)
            rewards_buf.append(reward)
            values_buf.append(value.item())
            dones_buf.append(float(done))
            log_probs_buf.append(log_probs.squeeze()[action].unsqueeze(0))

            current_episode_reward += reward
            state = next_state if not done else cartpole_reset()

            if done:
                episode_rewards.append(current_episode_reward)
                current_episode_reward = 0.0
                ep_count += 1
                if ep_count >= n_episodes:
                    break

        # Bootstrap next value
        s_t = torch.FloatTensor(state).unsqueeze(0).to(device)
        with torch.no_grad():
            _, next_val = ac(s_t)
        next_value = next_val.item()

        # Compute GAE advantages
        advantages = compute_gae(rewards_buf, values_buf, next_value, dones_buf, gamma, lam)
        advantages_t = torch.FloatTensor(advantages).to(device)

        # Compute value targets = advantages + values
        value_targets = advantages + np.array(values_buf)
        value_targets_t = torch.FloatTensor(value_targets).to(device)

        # Normalize advantages
        advantages_t = (advantages_t - advantages_t.mean()) / (advantages_t.std() + 1e-8)

        # Recompute log_probs and values through network (with gradients)
        states_t = torch.FloatTensor(np.array(states_buf)).to(device)
        actions_t = torch.LongTensor(actions_buf).to(device)
        log_probs_net, values_net = ac(states_t)

        selected_log_probs = log_probs_net.gather(1, actions_t.unsqueeze(1)).squeeze(1)

        # Actor loss: -mean(log_pi * advantage)
        actor_loss = -(selected_log_probs * advantages_t.detach()).mean()

        # Critic loss: MSE between V(s) and value targets
        critic_loss = F.mse_loss(values_net, value_targets_t.detach())

        # Entropy bonus
        entropy = -(log_probs_net.exp() * log_probs_net).sum(dim=-1).mean()
        entropy_log.append(entropy.item())

        # Combined A2C loss
        total_loss = actor_loss + value_coeff * critic_loss - entropy_coeff * entropy

        optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(ac.parameters(), max_norm=0.5)
        optimizer.step()

    return episode_rewards, entropy_log


torch.manual_seed(42); np.random.seed(42)
a2c_rewards, a2c_entropy = run_a2c(n_episodes=400)
print(f'A2C final performance (last 50 eps): {np.mean(a2c_rewards[-50:]):.1f}')

## Real-World Example 1: Continuous Control — Pendulum Swing-Up

Pendulum physics: theta'' = -(3g/2l)sin(theta) + u/(ml^2).  
Gaussian policy outputs torque. Train actor-critic to swing up and balance.

In [ ]:
class PendulumEnv:
    """1D pendulum swing-up. State: [theta, theta_dot]. Action: torque in [-2, 2]."""

    def __init__(self, dt: float = 0.05, g: float = 9.8, l: float = 1.0, m: float = 1.0):
        self.dt = dt; self.g = g; self.l = l; self.m = m
        self.max_torque = 2.0
        self.max_theta_dot = 8.0
        self.reset()

    def reset(self) -> np.ndarray:
        # Start near bottom (theta ~ pi = hanging down)
        self.theta = np.pi + np.random.uniform(-0.1, 0.1)
        self.theta_dot = np.random.uniform(-0.1, 0.1)
        return self._obs()

    def _obs(self) -> np.ndarray:
        # Encode as [cos(theta), sin(theta), theta_dot] for continuity
        return np.array([np.cos(self.theta), np.sin(self.theta), self.theta_dot / self.max_theta_dot])

    def step(self, action: float):
        u = np.clip(action, -self.max_torque, self.max_torque)
        # Physics: theta'' = (3g/2l)*sin(theta) + 3/(ml^2)*u
        theta_acc = (3 * self.g / (2 * self.l)) * np.sin(self.theta) + (3.0 / (self.m * self.l**2)) * u
        self.theta_dot = np.clip(self.theta_dot + self.dt * theta_acc, -self.max_theta_dot, self.max_theta_dot)
        self.theta += self.dt * self.theta_dot
        # Normalize theta to [-pi, pi]
        self.theta = ((self.theta + np.pi) % (2 * np.pi)) - np.pi
        # Reward: negative angle from upright (0) and energy cost
        reward = -(self.theta**2 + 0.1 * self.theta_dot**2 + 0.001 * u**2)
        return self._obs(), reward, False  # Never terminates


class ContinuousActorCritic(nn.Module):
    """Actor-critic with Gaussian policy for continuous actions."""

    def __init__(self, state_dim: int, hidden: int = 64):
        super().__init__()
        self.base = nn.Sequential(nn.Linear(state_dim, hidden), nn.Tanh(),
                                  nn.Linear(hidden, hidden), nn.Tanh())
        self.mu_head = nn.Linear(hidden, 1)    # Mean of Gaussian
        self.log_std = nn.Parameter(torch.zeros(1))  # Learnable log std
        self.value_head = nn.Linear(hidden, 1)  # V(s)

    def forward(self, x: torch.Tensor):
        h = self.base(x)
        mu = 2.0 * torch.tanh(self.mu_head(h))  # Action in [-2, 2]
        std = self.log_std.exp().clamp(0.01, 1.0)
        value = self.value_head(h).squeeze(-1)
        return mu, std, value


def run_pendulum_ac(n_episodes: int = 200, n_steps: int = 50, gamma: float = 0.99,
                    lr: float = 3e-4, entropy_coeff: float = 0.005) -> list:
    """Actor-critic on Pendulum swing-up task."""
    env = PendulumEnv()
    ac = ContinuousActorCritic(state_dim=3).to(device)
    optimizer = optim.Adam(ac.parameters(), lr=lr)
    rewards_hist = []

    for ep in range(n_episodes):
        state = env.reset()
        total_reward = 0.0
        states_buf, actions_buf, rewards_buf, values_buf = [], [], [], []

        for _ in range(n_steps):
            s_t = torch.FloatTensor(state).unsqueeze(0).to(device)
            with torch.no_grad():
                mu, std, value = ac(s_t)
            dist = torch.distributions.Normal(mu, std)
            action = dist.sample().squeeze().item()

            next_state, reward, done = env.step(action)
            states_buf.append(state.copy())
            actions_buf.append(action)
            rewards_buf.append(reward)
            values_buf.append(value.item())
            total_reward += reward
            state = next_state

        rewards_hist.append(total_reward / n_steps)

        # Compute returns and advantages (no dones: pendulum never terminates)
        advantages = compute_gae(rewards_buf, values_buf, values_buf[-1],
                                  [0.0] * n_steps, gamma=gamma, lam=0.95)
        advantages_t = torch.FloatTensor(advantages).to(device)
        advantages_t = (advantages_t - advantages_t.mean()) / (advantages_t.std() + 1e-8)
        value_targets_t = torch.FloatTensor(advantages + np.array(values_buf)).to(device)

        states_t = torch.FloatTensor(np.array(states_buf)).to(device)
        actions_t = torch.FloatTensor(actions_buf).to(device)

        mu_net, std_net, values_net = ac(states_t)
        dist_net = torch.distributions.Normal(mu_net.squeeze(), std_net)
        log_probs_net = dist_net.log_prob(actions_t)
        entropy = dist_net.entropy().mean()

        actor_loss = -(log_probs_net * advantages_t.detach()).mean()
        critic_loss = F.mse_loss(values_net, value_targets_t.detach())
        loss = actor_loss + 0.5 * critic_loss - entropy_coeff * entropy

        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(ac.parameters(), 0.5)
        optimizer.step()

    return rewards_hist


torch.manual_seed(42); np.random.seed(42)
pendulum_rewards = run_pendulum_ac(n_episodes=200)
print(f'Pendulum AC: mean reward per step (last 50 eps) = {np.mean(pendulum_rewards[-50:]):.3f}')

## Real-World Example 2: Multi-Step Returns — n-Step Advantage Estimates

Compare 1-step TD, 5-step, 20-step advantage estimates.  
Larger n reduces bias but increases variance.

In [ ]:
def compute_nstep_advantages(
    rewards: list,
    values: list,
    next_value: float,
    n: int,
    gamma: float = 0.99,
) -> np.ndarray:
    """Compute n-step return advantages. G_{t:t+n} = r_t + ... + gamma^{n-1}*r_{t+n-1} + gamma^n*V(s_{t+n})."""
    T = len(rewards)
    values_ext = list(values) + [next_value]
    advantages = np.zeros(T)

    for t in range(T):
        # n-step return from t
        G = 0.0
        end = min(t + n, T)
        for k in range(t, end):
            G += (gamma ** (k - t)) * rewards[k]
        # Bootstrap from V(s_{t+n}) if we did not reach end of buffer
        if t + n < T:
            G += (gamma ** n) * values_ext[t + n]
        else:
            G += (gamma ** (T - t)) * next_value
        advantages[t] = G - values_ext[t]

    return advantages


def run_a2c_nstep(n_steps_rollout: int = 20, n_bootstrap: int = 1,
                   n_episodes: int = 300, lr: float = 3e-4) -> list:
    """A2C with configurable n-step advantage estimates."""
    ac = ActorCriticNetwork(4, 2).to(device)
    optimizer = optim.Adam(ac.parameters(), lr=lr)
    rewards = []; state = cartpole_reset(); ep_reward = 0.0; ep_count = 0

    while ep_count < n_episodes:
        states_buf, acts_buf, rwds_buf, vals_buf, done_buf = [], [], [], [], []
        for _ in range(n_steps_rollout):
            s_t = torch.FloatTensor(state).unsqueeze(0).to(device)
            with torch.no_grad():
                lp, v = ac(s_t)
            probs = lp.exp().squeeze().cpu().numpy()
            action = np.random.choice(2, p=probs)
            ns, r, done = cartpole_step(state, action)
            states_buf.append(state.copy()); acts_buf.append(action)
            rwds_buf.append(r); vals_buf.append(v.item()); done_buf.append(float(done))
            ep_reward += r; state = ns if not done else cartpole_reset()
            if done:
                rewards.append(ep_reward); ep_reward = 0.0; ep_count += 1
                if ep_count >= n_episodes: break

        s_t = torch.FloatTensor(state).unsqueeze(0).to(device)
        with torch.no_grad(): _, nv = ac(s_t)
        advs = compute_nstep_advantages(rwds_buf, vals_buf, nv.item(), n_bootstrap, gamma=0.99)
        advs_t = torch.FloatTensor(advs).to(device)
        advs_t = (advs_t - advs_t.mean()) / (advs_t.std() + 1e-8)
        vtgt = torch.FloatTensor(advs + np.array(vals_buf)).to(device)

        st = torch.FloatTensor(np.array(states_buf)).to(device)
        at = torch.LongTensor(acts_buf).to(device)
        lp_n, v_n = ac(st)
        slp = lp_n.gather(1, at.unsqueeze(1)).squeeze(1)
        ent = -(lp_n.exp() * lp_n).sum(dim=-1).mean()
        loss = -(slp * advs_t.detach()).mean() + 0.5 * F.mse_loss(v_n, vtgt.detach()) - 0.01 * ent
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(ac.parameters(), 0.5)
        optimizer.step()

    return rewards


n_bootstrap_values = [1, 5, 20]
nstep_rewards_dict = {}
for n_boot in n_bootstrap_values:
    torch.manual_seed(42); np.random.seed(42)
    nstep_rewards_dict[n_boot] = run_a2c_nstep(n_steps_rollout=20, n_bootstrap=n_boot, n_episodes=300)
    print(f'n-step={n_boot:2d}: mean reward (last 50) = {np.mean(nstep_rewards_dict[n_boot][-50:]):.1f}')

## Real-World Example 3: Entropy Regularization

Entropy bonus H(pi) added to actor loss prevents premature convergence to deterministic policy.  
Compare beta in {0, 0.01, 0.05} on CartPole.

In [ ]:
def run_a2c_entropy(
    n_episodes: int = 300,
    n_steps: int = 20,
    gamma: float = 0.99,
    lam: float = 0.95,
    lr: float = 3e-4,
    entropy_coeff: float = 0.01,
    value_coeff: float = 0.5,
) -> tuple:
    """A2C with configurable entropy coefficient. Returns (rewards, entropy_curve)."""
    ac = ActorCriticNetwork(4, 2).to(device)
    optimizer = optim.Adam(ac.parameters(), lr=lr)
    rewards = []; entropy_log = []
    state = cartpole_reset(); ep_reward = 0.0; ep_count = 0

    while ep_count < n_episodes:
        states_buf, acts_buf = [], []
        rwds_buf, vals_buf, done_buf = [], [], []

        for _ in range(n_steps):
            s_t = torch.FloatTensor(state).unsqueeze(0).to(device)
            with torch.no_grad(): lp, v = ac(s_t)
            probs = lp.exp().squeeze().cpu().numpy()
            action = np.random.choice(2, p=probs)
            ns, r, done = cartpole_step(state, action)
            states_buf.append(state.copy()); acts_buf.append(action)
            rwds_buf.append(r); vals_buf.append(v.item()); done_buf.append(float(done))
            ep_reward += r
            state = ns if not done else cartpole_reset()
            if done:
                rewards.append(ep_reward); ep_reward = 0.0; ep_count += 1
                if ep_count >= n_episodes: break

        s_t = torch.FloatTensor(state).unsqueeze(0).to(device)
        with torch.no_grad(): _, nv = ac(s_t)

        advs = compute_gae(rwds_buf, vals_buf, nv.item(), done_buf, gamma, lam)
        advs_t = torch.FloatTensor(advs).to(device)
        advs_t = (advs_t - advs_t.mean()) / (advs_t.std() + 1e-8)
        vtgt = torch.FloatTensor(advs + np.array(vals_buf)).to(device)

        st = torch.FloatTensor(np.array(states_buf)).to(device)
        at = torch.LongTensor(acts_buf).to(device)
        lp_n, v_n = ac(st)
        slp = lp_n.gather(1, at.unsqueeze(1)).squeeze(1)
        ent = -(lp_n.exp() * lp_n).sum(dim=-1).mean()
        entropy_log.append(ent.item())

        loss = -(slp * advs_t.detach()).mean() + value_coeff * F.mse_loss(v_n, vtgt.detach()) \
               - entropy_coeff * ent
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(ac.parameters(), 0.5)
        optimizer.step()

    return rewards, entropy_log


entropy_coeffs = [0.0, 0.01, 0.05]
entropy_results = {}

for beta in entropy_coeffs:
    torch.manual_seed(42); np.random.seed(42)
    r, elog = run_a2c_entropy(n_episodes=300, entropy_coeff=beta)
    entropy_results[beta] = (r, elog)
    print(f'beta={beta:.2f}: reward (last 50)={np.mean(r[-50:]):.1f}, '
          f'final entropy={np.mean(elog[-10:]):.3f}')

## Comparison: REINFORCE vs A2C vs A2C+Entropy

Visualize learning curves, entropy evolution, and n-step advantage trade-offs.

In [ ]:
def smooth(arr, w=15):
    if len(arr) < w:
        return np.array(arr)
    return np.convolve(arr, np.ones(w) / w, mode='valid')


fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Plot 1: A2C vs Tabular AC learning curves ---
ax = axes[0]
ax.plot(smooth(a2c_rewards), label='A2C (neural, GAE)', color='blue')
ax.plot(smooth(tab_ac_rewards), label='Tabular AC (GridWorld)', color='orange')
ax.set_title('Actor-Critic: Tabular vs Neural')
ax.set_xlabel('Episode'); ax.set_ylabel('Episode Reward (smoothed)')
ax.legend()

# --- Plot 2: Entropy regularization effect ---
ax = axes[1]
colors = ['red', 'blue', 'green']
for beta, col in zip(entropy_coeffs, colors):
    r, elog = entropy_results[beta]
    ax.plot(smooth(r), color=col, label=f'beta={beta}')
ax.set_title('A2C: Effect of Entropy Regularization')
ax.set_xlabel('Episode'); ax.set_ylabel('Episode Reward (smoothed)')
ax.legend()

# --- Plot 3: n-step advantage trade-off ---
ax = axes[2]
nstep_colors = ['blue', 'orange', 'green']
for n_boot, col in zip(n_bootstrap_values, nstep_colors):
    ax.plot(smooth(nstep_rewards_dict[n_boot]), color=col, label=f'n={n_boot}')
ax.set_title('n-Step Returns: Bias vs Variance')
ax.set_xlabel('Episode'); ax.set_ylabel('Episode Reward (smoothed)')
ax.legend()

plt.suptitle('Actor-Critic Methods Comparison', fontsize=13)
plt.tight_layout()
plt.savefig('/tmp/actor_critic_comparison.png', dpi=80, bbox_inches='tight')
plt.close()
print('Plot saved to /tmp/actor_critic_comparison.png')

print('\n=== Final Performance Summary ===')
print(f'{"Method":<30} {"Mean Reward (last 50)":<25}')
print('-' * 55)
print(f'{"Tabular AC (GridWorld)":<30} {np.mean(tab_ac_rewards[-50:]):.2f}')
print(f'{"Neural A2C (CartPole)":<30} {np.mean(a2c_rewards[-50:]):.1f}')
for beta in entropy_coeffs:
    r, _ = entropy_results[beta]
    print(f'{f"A2C entropy beta={beta}":<30} {np.mean(r[-50:]):.1f}')

## Key Takeaways

**Core idea:** Actor-Critic combines a policy network (actor) with a value function (critic). The critic provides a per-step advantage estimate A(s,a) = r + V(s') - V(s), which is lower variance than Monte Carlo returns, enabling faster convergence.

**Variants and when to use:**

| Method | Advantage | Update Frequency | When to Use |
|--------|-----------|-----------------|-------------|
| Tabular AC | V-table TD error | Per step | Discrete, small state spaces |
| A2C (GAE, lam=0.95) | GAE n-step | Per rollout | Most practical tasks |
| A3C | GAE async | Per step (async) | Multicore CPU training |
| PPO | Clipped surrogate | Mini-batch epochs | RLHF, production RL |
| SAC | Soft advantage | Off-policy steps | Real robotics, sparse rewards |

**Common failure modes:**
- Critic not trained before actor: random advantage estimates; policy diverges immediately
- Advantages not normalized: large advantage scale causes gradient explosion; always normalize
- Entropy collapse: policy becomes deterministic too early; add beta=0.01 entropy coefficient

**Related concepts:**
- [09-policy-gradient](./09-policy-gradient.ipynb) — REINFORCE without critic; A2C extends it
- [08-deep-q-networks](./08-deep-q-networks.ipynb) — pure critic approach; actor-critic combines both
- [06-q-learning](./06-q-learning.ipynb) — tabular value learning that critic builds on

## Exercises

1. **GAE lambda sweep:** Run A2C with lambda in {0, 0.3, 0.7, 0.95, 1.0}. Plot mean reward and advantage variance as functions of lambda. Identify the optimal lambda for CartPole.
2. **Separate networks:** Modify A2C to use completely separate actor and critic networks (no shared base). Compare convergence speed and final performance versus shared-base version.
3. **Critic warmup:** Add a 50-episode pre-training phase where only the critic is updated (actor frozen). Does pre-training the critic improve actor convergence speed?
4. **Pendulum visualization:** Plot theta vs time for the trained continuous actor-critic. Does the agent successfully swing up from theta=pi (hanging) to theta=0 (upright)? Plot the torque output alongside theta.